# Robust Hierarchical Allocation — Synthetic Pre-registration Demo

**Synthetic data only.** This notebook executes a reduced factorial grid consistent with the pre-registered design in `RESEARCH_NOTE_UPGRADED.md`. No real-market claims.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
from data.generate import make_synthetic_panel
from backtest.walk_forward import walk_forward
from metrics.evaluation import portfolio_stats, frobenius, effective_bets, block_bootstrap_sharpe

pd.set_option('display.float_format', lambda x: f'{x:.4f}')
print('Imports OK')

In [ ]:
def run_cell(n_assets, corr_type, lookback, seed, cost_bps=5.0, n_periods=150):
    rets, meta = make_synthetic_panel(n_assets, n_periods, corr_type, lookback, seed)
    res = walk_forward(rets, meta['true_sigmas'], lookback, cost_bps=cost_bps)
    rows = []
    for m in res['returns'].columns:
        stats = portfolio_stats(res['returns'][m])
        lo, hi = block_bootstrap_sharpe(res['returns'][m], n_boot=80, seed=seed)
        rows.append({
            'n_assets': n_assets,
            'corr_type': corr_type,
            'lookback': lookback,
            'seed': seed,
            'method': m,
            'mean_frobenius': frobenius(res['errors'][m]),
            'eff_bets': effective_bets(res['weights'][m]),
            'sharpe_lo': lo,
            'sharpe_hi': hi,
            **stats,
        })
    return pd.DataFrame(rows)

# Reduced primary grid (still covers N/T, correlation structure, seeds)
cells = []
for n in [8, 20]:
    for corr in ['low', 'moderate', 'high']:
        for T in [36, 60]:
            for seed in [1, 2]:
                cells.append(run_cell(n, corr, T, seed))
df = pd.concat(cells, ignore_index=True)
print(f'Completed {len(df)} result rows')
df.head()

In [ ]:
# Primary diagnostic: estimation error by N and method
err = df.groupby(['n_assets', 'method'])['mean_frobenius'].mean().unstack()
print('Mean Frobenius estimation error')
display(err.round(4))

In [ ]:
# Stability diagnostics
stab = df.groupby('method')[['eff_bets', 'ann_vol', 'sharpe']].mean()
print('Average effective bets, volatility, Sharpe (net of 5 bps)')
display(stab.round(3))

In [ ]:
# Example of bootstrap intervals for one cell
one = df[(df.n_assets == 20) & (df.corr_type == 'high') & (df.lookback == 36)]
print('Sharpe 90% block-bootstrap intervals (N=20, high corr, T=36)')
display(one.groupby('method')[['sharpe', 'sharpe_lo', 'sharpe_hi']].mean().round(3))

## Interpretation note (pre-registered)

Any ranking observed above is conditional on the synthetic DGP, the chosen N/T ratios, and the 5 bp cost. A method is not declared superior unless the pre-registered criteria in the research note are met on the full grid. Negative or mixed outcomes are expected and informative.